# Exp 27: Smaller image sizes

Card: [exp27_small_images.md](../../experiments/exp27_small_images.md)


In [ ]:
import os

import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, precision_recall_curve

from core.config import CLASSES, ROOT
from core.cv import CVResult, Config, CrossValidator
from core.gradcam import GradCamFigures
from core.tracking import Experiment

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-small-images-smoke",
        "epochs": 1,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
    N_SAMPLE = 2  # Cat, Dog images per comparison figure
else:
    OVERRIDES = {}
    N_SAMPLE = 16  # exp19 sample
SEED = 0  # exp19 sample
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
CONFIGS = {
    f"{size} px": Config.from_toml(
        CONFIG_DIR / "exp27_small_images" / f"resnet18_{size}.toml", **OVERRIDES
    )
    for size in (128, 64)
}
EXP20 = Config.from_toml(CONFIG_DIR / "exp20_coarse_dropout_non_vip" / "resnet18.toml")
EXP16 = Config.from_toml(
    CONFIG_DIR / "exp16_downsampling" / "resnet18.toml"
)  # exp19 rows drawn from exp16 OOF
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp27_small_images"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode
SERIES = {"224 px": "#2a78d6", "128 px": "#1baf7a", "64 px": "#eb6834"}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)


## Training


In [ ]:
results = {"224 px": CVResult.load(EXP20)}
results.update({name: CrossValidator(cfg).run() for name, cfg in CONFIGS.items()})
figures = {}

## Grad-CAM Cat, Dog: 224 vs 128 vs 64 px


In [ ]:
cams = {"224 px": GradCamFigures(EXP20)} | {
    name: GradCamFigures(cfg) for name, cfg in CONFIGS.items()
}
exp16 = GradCamFigures(EXP16)
for cls in ["Cat", "Dog"]:
    # exp19 rows; smoke OOF holds few rows
    rows = (cams["64 px"] if SMOKE else exp16).positive_rows(cls, N_SAMPLE, SEED)
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_compare.png"] = GradCamFigures.compare(cams, cls, rows)
plt.show()

## Precision-recall curves


In [ ]:
fig, axes = plt.subplots(
    1, len(CLASSES), figsize=(3.4 * len(CLASSES), 3.6), sharey=True
)
for cls, ax in zip(CLASSES, axes):
    for name, r in results.items():
        y, p = r.oof[cls], r.oof[f"p_{cls}"]
        precision, recall, _ = precision_recall_curve(y, p)
        ax.plot(
            recall,
            precision,
            color=SERIES[name],
            lw=2,
            drawstyle="steps-post",
            label=f"{name}, AP {average_precision_score(y, p):.3f}",
        )
    ax.set(
        title=f"{cls} ({int(y.sum())} positives)",
        xlabel="recall",
        xlim=(0, 1.02),
        ylim=(0, 1.05),
    )
    ax.legend(loc="lower left", alignment="left")
axes[0].set_ylabel("precision")
fig.tight_layout()
figures[f"{PREFIX}_pr_curves.png"] = fig
plt.show()

In [ ]:
with Experiment(CONFIGS["64 px"].experiment).run("summary", {"val_mode": "fast"}) as run:
    if SAVE_FIGURES:
        run.log_figures(figures, FIGURES)